# Building a RAG System Locally

Companion notebook for **RAG from Basics to Advanced (Part 8)** — [full series](https://builder.aws.com/community/@pabbico)

This notebook builds a complete RAG pipeline from scratch on your own machine. No cloud services, no API keys, no framework abstractions hiding the parts that matter.

**What gets built, in order:**

| Step | What happens |
|---|---|
| 1 | Load and extract text from PDF and DOCX files |
| 2 | Split documents into chunks |
| 3 | Convert chunks into embeddings |
| 4 | Store and index them in ChromaDB |
| 5 | Retrieve relevant chunks for a query |
| 6 | Rerank retrieved chunks with a cross-encoder |
| 7 | Generate a grounded answer with a local LLM |

**Before running:** make sure Ollama is running and the model is pulled (`ollama pull llama3.1:8b`). See the README for full setup.

**Run this notebook top to bottom.** Several cells depend on variables defined earlier — running out of order will produce confusing results rather than clean errors.

---
## Configuration

Everything tunable lives here, in one place. Changing a value below and re-running the notebook from Section 1 is how you experiment — which matters, because almost every number here is a trade-off rather than a correct answer.

In [1]:
from pathlib import Path

# --- Documents ---
DOCS_DIR = Path("../data/sample_docs")

# --- Chunking ---
CHUNK_SIZE = 500       # target characters per chunk
OVERLAP = 100          # characters repeated between chunks when a hard split is needed
MIN_CHUNK_SIZE = 150   # chunks smaller than this are dropped as noise

# --- Embedding ---
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# --- Vector store ---
CHROMA_PATH = "../chroma_db"
COLLECTION_NAME = "pawan_docs"

# --- Reranking ---
RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RETRIEVE_K = 20        # candidates from stage 1 (bi-encoder)
FINAL_K = 5            # chunks kept after stage 2 (cross-encoder)

# --- Generation ---
OLLAMA_URL = "http://localhost:11434/api/generate"
LLM_MODEL = "llama3.1:8b"
TEMPERATURE = 0.1      # low: stick to the context, don't get creative

print("Config loaded.")

Config loaded.


---
# 1. Loading Documents

Before anything else, text has to come out of the source files. This step looks trivial and is where the most damaging bugs hide — because when extraction silently drops content, nothing downstream raises an error. Retrieval just quietly can't find things.

In [2]:
from pypdf import PdfReader
from docx import Document

files = sorted([f for f in DOCS_DIR.iterdir() if f.suffix in [".pdf", ".docx"]])

print(f"Found {len(files)} files:\n")
for f in files:
    print(f"  {f.name:40} {f.stat().st_size / 1024:>6.1f} KB")

Found 6 files:

  Customer_Support_FAQ.pdf                   88.0 KB
  Employee_Handbook.docx                     16.3 KB
  IT_Security_Policy.pdf                     80.2 KB
  Product_Catalogue_2025-26.docx             14.5 KB
  Refund_and_Returns_Policy.pdf              76.9 KB
  Travel_and_Expense_Policy.docx             14.8 KB


### Extraction functions

Two details in the code below are worth understanding, because both are easy to get wrong:

**Page numbers only work for PDFs.** A PDF stores page boundaries as fixed structure, so `pypdf` reports them reliably. A `.docx` doesn't store page numbers at all — pagination is decided at render time and depends on fonts, margins, and which application opens it. So PDF chunks can carry a page number in their citation; DOCX chunks can't. That's a limitation of the format, not a bug to fix.

**Word tables are skipped by default.** Iterating `doc.paragraphs` returns paragraph text only — table content is ignored, with no error and no warning. In this dataset that would silently drop the entire spare parts table while character counts still looked perfectly healthy. Tables are extracted explicitly via `doc.tables` for exactly that reason.

In [3]:
def extract_pdf(path):
    """Extract text from a PDF, page by page."""
    reader = PdfReader(path)
    pages = []
    for i, page in enumerate(reader.pages, start=1):
        pages.append({"page": i, "text": page.extract_text() or ""})
    return pages


def extract_docx(path):
    """Extract text from a DOCX, including table content."""
    doc = Document(path)
    blocks = []

    for para in doc.paragraphs:
        if para.text.strip():
            blocks.append(para.text.strip())

    # Tables are NOT included in doc.paragraphs - pull them separately
    for table in doc.tables:
        for row in table.rows:
            cells = [c.text.strip() for c in row.cells if c.text.strip()]
            if cells:
                blocks.append(" | ".join(cells))

    return blocks

In [4]:
documents = []

for f in files:
    if f.suffix == ".pdf":
        pages = extract_pdf(f)
        documents.append({
            "source": f.name,
            "type": "pdf",
            "page_count": len(pages),
            "text": "\n".join(p["text"] for p in pages),
        })
    else:
        blocks = extract_docx(f)
        documents.append({
            "source": f.name,
            "type": "docx",
            "page_count": None,
            "text": "\n".join(blocks),
        })

for d in documents:
    print(f"{d['source']:40} {d['type']:5} {len(d['text']):>7,} chars")

Customer_Support_FAQ.pdf                 pdf     9,327 chars
Employee_Handbook.docx                   docx   13,393 chars
IT_Security_Policy.pdf                   pdf    11,186 chars
Product_Catalogue_2025-26.docx           docx    8,179 chars
Refund_and_Returns_Policy.pdf            pdf     8,873 chars
Travel_and_Expense_Policy.docx           docx    9,462 chars


### Verify the extraction

Character counts prove that *something* came out. They don't prove it's usable — PDF extraction in particular can produce garbled text that still counts as thousands of characters. Reading a sample by eye is the cheapest debugging step available, and skipping it is how people end up debugging their embedding model for a problem that started here.

Note also that file size tells you nothing about text content: in this dataset an 88 KB PDF yields ~9,300 characters while a 16 KB DOCX yields ~13,400.

In [5]:
sample = documents[0]
print(f"=== {sample['source']} ===\n")
print(sample["text"][:700])
print("\n... [truncated]")

=== Customer_Support_FAQ.pdf ===

Customer Support Handbook and FAQ
Pawan Pvt Ltd — Customer Operations
Version 5.0 | Effective 1 February 2025
Table of Contents
1. About This Handbook
This handbook is the reference used by customer support agents at Pawan Pvt Ltd. It records 
the answers to the questions customers ask most often, together with the internal procedure for 
each.
Agents should answer from this handbook wherever a question is covered. Questions not 
covered here are escalated to a team lead rather than answered from personal judgement.
The handbook is updated monthly. Agents are notified of changes through the support team 
channel.
2. Contacting Support
2.1 Support Channels
Customers can reach support through 

... [truncated]


In [6]:
# Did the table extraction actually work? Check for a part code that only
# exists inside a Word table.
catalogue = [d for d in documents if "Catalogue" in d["source"]][0]

print("SP-1101 found?", "SP-1101" in catalogue["text"])
print()
for line in catalogue["text"].split("\n"):
    if line.startswith("SP-"):
        print(line)

SP-1101 found? True

SP-1101 | Mechanical seal kit, 25 mm | PX-1200 | 1,240
SP-1102 | Mechanical seal kit, 40 mm | PX-2400 | 1,680
SP-1103 | Mechanical seal kit, 50 mm | PX-3600, PX-3600S | 2,150
SP-2201 | Impeller, brass, 120 LPM | PX-1200 | 3,400
SP-2202 | Impeller, SS304, 240 LPM | PX-2400 | 6,900
SP-2203 | Impeller, SS316, 360 LPM | PX-3600 | 11,200
SP-3301 | Bearing set, light duty | PX-1200, SB-0750 | 890
SP-3302 | Bearing set, heavy duty | PX-3600, SB-3000 | 2,340
SP-4401 | Control card, CP-300 | CP-300 | 7,800
SP-4402 | Display module, CP-500V | CP-500V | 9,600
SP-5501 | Pressure transducer, 0-10 bar | CP-500V | 5,200
SP-5502 | Dry-run sensor probe | CP-100, CP-300 | 1,450


---
# 2. Chunking

Documents are too large to embed whole — embedding models have a token limit, and a single vector for a 7-page document averages away everything specific in it. So documents get split into chunks, and each chunk becomes one searchable unit.

The chunking code below went through three revisions during development. Both fixes are already applied here, but the reasons are worth knowing because you'll hit the same problems on your own documents:

**Revision 1 → 2: PDF line breaks aren't paragraph breaks.** Splitting on `\n` seemed reasonable until chunks started ending mid-sentence. PDF extraction inserts a line break at every *visual* line. `normalize_text()` rejoins those continuation lines and breaks only at genuine boundaries — headings, bullets, table rows, blank lines.

**Revision 2 → 3: character-index splits cut words in half.** Hard-splitting a long paragraph at `para[start:start+500]` turned "team lead" into "ad". Long paragraphs are now split on word boundaries, and overlap is measured in words rather than characters.

Two more things the code does deliberately:

- **Each chunk gets a `[Document Name]` header.** A small addition that meaningfully helps retrieval, because the source context gets encoded into the embedding itself.
- **Tables are kept separate from prose, and the header row is repeated.** Without this, a table gets glued onto whatever paragraph preceded it (diluting the embedding across two unrelated topics), and any chunk containing the middle of a table arrives without column names — leaving the LLM to guess whether `9,600` is a price or a part number.

In [7]:
import re

def normalize_text(text):
    """
    Rejoin false line breaks from PDF extraction.

    PDFs break lines visually, not semantically, so a paragraph arrives as
    several \n-separated lines. Real boundaries are blank lines, headings,
    bullets, and table rows.
    """
    paragraphs = []
    buffer = ""

    for line in text.split("\n"):
        stripped = line.strip()

        if not stripped:
            if buffer:
                paragraphs.append(buffer.strip())
                buffer = ""
            continue

        is_heading = bool(re.match(r"^\d+(\.\d+)*\s+[A-Z]", stripped))
        is_bullet = stripped.startswith(("\u2022", "-", "*"))
        is_table_row = "|" in stripped

        if is_heading or is_bullet or is_table_row:
            if buffer:
                paragraphs.append(buffer.strip())
            buffer = stripped
        else:
            buffer = f"{buffer} {stripped}" if buffer else stripped

    if buffer:
        paragraphs.append(buffer.strip())

    return paragraphs

In [8]:
def chunk_document(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Split one document into chunks, keeping tables intact."""
    paragraphs = normalize_text(doc["text"])

    chunks = []
    current = ""
    table_header = None

    for para in paragraphs:
        is_table_row = "|" in para

        # A table is starting and the current chunk holds prose.
        # Close it first - don't mix two unrelated topics in one embedding.
        if is_table_row and current and "|" not in current:
            chunks.append(current)
            current = ""

        # Remember the first row of a table as its header
        if is_table_row and table_header is None:
            table_header = para
        elif not is_table_row:
            table_header = None

        if len(current) + len(para) + 1 <= chunk_size:
            current = f"{current}\n{para}" if current else para
        else:
            if current:
                chunks.append(current)

            if is_table_row and table_header and table_header != para:
                # New chunk starts mid-table - repeat the header so the
                # columns still mean something on their own
                current = f"{table_header}\n{para}"
            elif len(para) > chunk_size:
                # Paragraph longer than chunk_size: split on word boundaries
                words = para.split()
                buf = ""
                for w in words:
                    if len(buf) + len(w) + 1 <= chunk_size:
                        buf = f"{buf} {w}" if buf else w
                    else:
                        chunks.append(buf)
                        tail = buf.split()[-(overlap // 6):]   # overlap in words
                        buf = " ".join(tail + [w])
                current = buf
            else:
                current = para

    if current:
        chunks.append(current)

    # Drop chunks too small to carry information (usually stray headings)
    chunks = [c for c in chunks if len(c) >= MIN_CHUNK_SIZE]

    doc_title = doc["source"].replace("_", " ").rsplit(".", 1)[0]
    return [f"[{doc_title}]\n{c}" for c in chunks]

In [9]:
all_chunks = []

for doc in documents:
    for i, ch in enumerate(chunk_document(doc)):
        all_chunks.append({
            "id": f"{doc['source']}::chunk_{i}",
            "text": ch,
            "source": doc["source"],
            "doc_type": doc["type"],
            "chunk_index": i,
        })

print(f"Total chunks: {len(all_chunks)}\n")

from collections import Counter
for src, n in sorted(Counter(c["source"] for c in all_chunks).items()):
    print(f"  {src:40} {n:>3} chunks")

lengths = [len(c["text"]) for c in all_chunks]
print(f"\nChunk length -> min: {min(lengths)}, max: {max(lengths)}, avg: {sum(lengths)//len(lengths)}")

Total chunks: 166

  Customer_Support_FAQ.pdf                  24 chunks
  Employee_Handbook.docx                    36 chunks
  IT_Security_Policy.pdf                    31 chunks
  Product_Catalogue_2025-26.docx            23 chunks
  Refund_and_Returns_Policy.pdf             23 chunks
  Travel_and_Expense_Policy.docx            29 chunks

Chunk length -> min: 180, max: 528, avg: 398


### Inspect the chunks

The **minimum** length is the number to watch. If the smallest chunks are 30–80 characters, they're bare headings with no content — and they'll still be embedded, retrieved, and handed to the LLM as pure noise. That's what `MIN_CHUNK_SIZE` exists to prevent.

Also check that chunk boundaries land between sentences rather than through them, and that consecutive chunks overlap (the second should begin with text that ended the first).

In [10]:
print("=== SMALLEST CHUNKS ===")
for c in sorted(all_chunks, key=lambda c: len(c["text"]))[:3]:
    print(f"\n[{len(c['text'])} chars] {c['id']}")
    print(c["text"])

print("\n\n=== FIRST TWO CHUNKS (check boundaries and overlap) ===")
for c in all_chunks[:2]:
    print(f"\n--- {c['id']} ({len(c['text'])} chars) ---")
    print(c["text"])

=== SMALLEST CHUNKS ===

[180 chars] Customer_Support_FAQ.pdf::chunk_15
[Customer Support FAQ]
6.1 What is the warranty period? Twelve months from delivery for most products. PX series pumps carry 24 months and CP series control panels carry 36 months.

[187 chars] Employee_Handbook.docx::chunk_11
[Employee Handbook]
or discrimination of any kind. Unauthorised disclosure of confidential information. Being under the influence of alcohol or prohibited substances during working hours.

[189 chars] Employee_Handbook.docx::chunk_30
[Employee Handbook]
7.1 Informal Resolution Employees are encouraged to raise concerns directly with their reporting manager in the first instance. Most concerns are resolved at this stage.


=== FIRST TWO CHUNKS (check boundaries and overlap) ===

--- Customer_Support_FAQ.pdf::chunk_0 (515 chars) ---
[Customer Support FAQ]
Version 5.0 | Effective 1 February 2025 Table of Contents 1. About This Handbook This handbook is the reference used by customer support agen

In [11]:
# Check the table fix: content should be free of surrounding prose,
# and every chunk holding table rows should carry the header row.
for c in all_chunks:
    if "SP-1101" in c["text"] or "SP-4402" in c["text"]:
        print(f"--- {c['id']} ---")
        print(c["text"])
        print()

--- Product_Catalogue_2025-26.docx::chunk_21 ---
[Product Catalogue 2025-26]
Part code | Description | Compatible with | Price (Rs)
SP-1101 | Mechanical seal kit, 25 mm | PX-1200 | 1,240
SP-1102 | Mechanical seal kit, 40 mm | PX-2400 | 1,680
SP-1103 | Mechanical seal kit, 50 mm | PX-3600, PX-3600S | 2,150
SP-2201 | Impeller, brass, 120 LPM | PX-1200 | 3,400
SP-2202 | Impeller, SS304, 240 LPM | PX-2400 | 6,900
SP-2203 | Impeller, SS316, 360 LPM | PX-3600 | 11,200
SP-3301 | Bearing set, light duty | PX-1200, SB-0750 | 890

--- Product_Catalogue_2025-26.docx::chunk_22 ---
[Product Catalogue 2025-26]
Part code | Description | Compatible with | Price (Rs)
SP-3302 | Bearing set, heavy duty | PX-3600, SB-3000 | 2,340
SP-4401 | Control card, CP-300 | CP-300 | 7,800
SP-4402 | Display module, CP-500V | CP-500V | 9,600
SP-5501 | Pressure transducer, 0-10 bar | CP-500V | 5,200
SP-5502 | Dry-run sensor probe | CP-100, CP-300 | 1,450



---
## 2.5 How This Chunker Got Here

The chunking function above looks arbitrary until you see what happens without each piece. This section runs the earlier, broken versions on the same documents so the failures are visible rather than described.

This is worth doing rather than skipping, because these aren't exotic bugs — they're what you will hit on your own documents, and none of them raises an error.

In [12]:
def chunk_v1(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Version 1: split on newlines, hard-cut anything too long."""
    paragraphs = [p.strip() for p in doc["text"].split("\n") if p.strip()]

    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = f"{current}\n{para}" if current else para
        else:
            if current:
                chunks.append(current)
            if len(para) > chunk_size:
                start = 0
                while start < len(para):
                    chunks.append(para[start:start + chunk_size])
                    start += chunk_size - overlap
                current = ""
            else:
                current = para
    if current:
        chunks.append(current)
    return chunks


def chunk_v2(doc, chunk_size=CHUNK_SIZE, overlap=OVERLAP):
    """Version 2: normalised line breaks and a source header,
    but still splits long paragraphs by character index."""
    paragraphs = normalize_text(doc["text"])

    chunks, current = [], ""
    for para in paragraphs:
        if len(current) + len(para) + 1 <= chunk_size:
            current = f"{current}\n{para}" if current else para
        else:
            if current:
                chunks.append(current)
            if len(para) > chunk_size:
                start = 0
                while start < len(para):
                    chunks.append(para[start:start + chunk_size])
                    start += chunk_size - overlap
                current = ""
            else:
                current = para
    if current:
        chunks.append(current)

    title = doc["source"].replace("_", " ").rsplit(".", 1)[0]
    return [f"[{title}]\n{c}" for c in chunks]

### Failure 1: chunks ending mid-sentence

PDF extraction inserts a line break at every *visual* line, not at every paragraph. Version 1 treats each of those as a paragraph boundary, so chunks end wherever the line happened to wrap.

Look at where the first chunk stops below.

In [13]:
faq = [d for d in documents if "FAQ" in d["source"]][0]

v1 = chunk_v1(faq)
v2 = chunk_v2(faq)

print("=== V1: split on raw newlines ===")
print(f"...{v1[0][-120:]}")
print("\n    ^ note where this ends\n")

print("=== V2: after normalize_text() ===")
print(f"...{v2[0][-120:]}")
print("\n    ^ ends on a complete sentence")

=== V1: split on raw newlines ===
...h the internal procedure for
each.
Agents should answer from this handbook wherever a question is covered. Questions not

    ^ note where this ends

=== V2: after normalize_text() ===
...[Customer Support FAQ]
Customer Support Handbook and FAQ Pawan Pvt Ltd — Customer Operations

    ^ ends on a complete sentence


### Failure 2: words cut in half

Version 2 fixed sentence boundaries but still hard-split long paragraphs at a character index. Splitting at exactly 500 characters lands wherever it lands — often inside a word.

The cell below finds chunks that begin with a word fragment.

In [14]:
import re as _re

print("=== V2: chunk starts (looking for broken words) ===\n")

broken = 0
for i, c in enumerate(chunk_v2(faq)[:8]):
    body = c.split("\n", 1)[1] if "\n" in c else c
    first_word = body.split()[0] if body.split() else ""
    # A real sentence start is capitalised, numeric, or a bullet
    looks_broken = bool(first_word) and first_word[0].islower()
    flag = "  <-- fragment" if looks_broken else ""
    broken += looks_broken
    print(f"[{i}] {first_word!r}{flag}")

print(f"\nChunks starting mid-word: {broken}")
print("\nThe final version splits on word boundaries instead, so this doesn't happen.")

=== V2: chunk starts (looking for broken words) ===

[0] 'Customer'
[1] 'Version'
[2] 'ad'  <-- fragment
[3] '2.1'
[4] '2.2'
[5] '2.3'
[6] '3.1'
[7] '3.2'

Chunks starting mid-word: 1

The final version splits on word boundaries instead, so this doesn't happen.


### Failure 3: chunks with no content in them

Every document starts with a title block. Left alone, that becomes its own chunk — 80 characters of document name and department, and nothing else.

These get embedded, indexed, and retrieved like any other chunk, feeding the LLM context that contains no information. `MIN_CHUNK_SIZE` drops them.

In [15]:
tiny = [c for c in chunk_v2(faq) if len(c) < MIN_CHUNK_SIZE]

print(f"V2 chunks below MIN_CHUNK_SIZE ({MIN_CHUNK_SIZE} chars): {len(tiny)}\n")
for c in tiny:
    print(f"[{len(c)} chars] {c!r}")

final = chunk_document(faq)
print(f"\nFinal version, smallest chunk: {min(len(c) for c in final)} chars")

V2 chunks below MIN_CHUNK_SIZE (150 chars): 1

[92 chars] '[Customer Support FAQ]\nCustomer Support Handbook and FAQ Pawan Pvt Ltd — Customer Operations'

Final version, smallest chunk: 180 chars


### Failure 4: tables glued to unrelated prose

This one only shows up in the product catalogue, and it's the most damaging of the four because it degrades retrieval quietly rather than looking broken.

Without table handling, the spare parts table gets appended to whatever paragraph preceded it — leaving one chunk carrying two unrelated topics, whose embedding is an average of both. And when the table spills into a second chunk, that chunk arrives with no header row, so `9,600` could be a price, a part number, or a quantity.

In [16]:
cat_doc = [d for d in documents if "Catalogue" in d["source"]][0]

print("=== V2: no table handling ===\n")
for c in chunk_v2(cat_doc):
    if "SP-1101" in c or "SP-4402" in c:
        print(c)
        print("-" * 70)

print("\n\n=== FINAL: tables separated, header repeated ===\n")
for c in chunk_document(cat_doc):
    if "SP-1101" in c or "SP-4402" in c:
        print(c)
        print("-" * 70)

=== V2: no table handling ===

[Product Catalogue 2025-26]
Part code | Description | Compatible with | Price (Rs)
SP-1101 | Mechanical seal kit, 25 mm | PX-1200 | 1,240
SP-1102 | Mechanical seal kit, 40 mm | PX-2400 | 1,680
SP-1103 | Mechanical seal kit, 50 mm | PX-3600, PX-3600S | 2,150
SP-2201 | Impeller, brass, 120 LPM | PX-1200 | 3,400
SP-2202 | Impeller, SS304, 240 LPM | PX-2400 | 6,900
SP-2203 | Impeller, SS316, 360 LPM | PX-3600 | 11,200
SP-3301 | Bearing set, light duty | PX-1200, SB-0750 | 890
----------------------------------------------------------------------
[Product Catalogue 2025-26]
SP-3302 | Bearing set, heavy duty | PX-3600, SB-3000 | 2,340
SP-4401 | Control card, CP-300 | CP-300 | 7,800
SP-4402 | Display module, CP-500V | CP-500V | 9,600
SP-5501 | Pressure transducer, 0-10 bar | CP-500V | 5,200
SP-5502 | Dry-run sensor probe | CP-100, CP-300 | 1,450
----------------------------------------------------------------------


=== FINAL: tables separated, header repeated 

### The four versions side by side

Watch the **minimum** chunk length in particular. Version 2's minimum of around 80 characters is the title-only chunk; the final version's floor is the `MIN_CHUNK_SIZE` filter doing its job.

In [17]:
def summarise(name, chunker):
    total = []
    for d in documents:
        total.extend(chunker(d))
    lens = [len(c) for c in total]
    print(f"{name:28} {len(total):>4} chunks   "
          f"min {min(lens):>4}   avg {sum(lens)//len(lens):>4}   max {max(lens):>4}")

print(f"{'VERSION':28} {'COUNT':>10}   {'MIN':>8}   {'AVG':>7}   {'MAX':>7}")
print("-" * 80)
summarise("v1: raw newline split", chunk_v1)
summarise("v2: normalised + header", chunk_v2)
summarise("final: words + tables", chunk_document)

VERSION                           COUNT        MIN       AVG       MAX
--------------------------------------------------------------------------------
v1: raw newline split         138 chunks   min  133   avg  435   max  500
v2: normalised + header       181 chunks   min   80   avg  370   max  528
final: words + tables         166 chunks   min  180   avg  398   max  528


None of the four failures above throws an exception. Version 1 produces a perfectly reasonable-looking 138 chunks with healthy character counts. You would only discover the problems by reading the output — which is the actual lesson here, and the reason this notebook prints chunks rather than trusting the counts.

---
# 3. Embeddings

An embedding model converts text into a fixed-length vector of numbers. Text with similar meaning lands close together in that vector space — which is what makes semantic search possible at all.

`all-MiniLM-L6-v2` produces 384-dimensional vectors and runs comfortably on CPU. The first run downloads roughly 90 MB.

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np

embedder = SentenceTransformer(EMBEDDING_MODEL)

print(f"Model:      {EMBEDDING_MODEL}")
print(f"Dimensions: {embedder.get_embedding_dimension()}")
print(f"Max tokens: {embedder.max_seq_length}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model:      all-MiniLM-L6-v2
Dimensions: 384
Max tokens: 256


### The token limit is a hard constraint on chunk size

The embedding model caps at 256 tokens. Anything longer is **truncated silently** — no error, no warning, and the chunk text still prints out complete. Only its *embedding* is incomplete, which degrades retrieval in a way that's genuinely hard to trace back to its cause.

This is why chunk size has an upper bound that has nothing to do with the LLM's context window. Worth measuring rather than assuming: this dataset runs at roughly 2.75 characters per token — well below the usual English average of ~4 — because the documents are dense with product codes, numbers, and punctuation that the tokenizer splits aggressively. That puts the practical ceiling here at around 700 characters per chunk.

In [19]:
tokenizer = embedder.tokenizer
token_counts = [len(tokenizer.encode(c["text"])) for c in all_chunks]

print(f"Token count -> min: {min(token_counts)}, max: {max(token_counts)}, avg: {sum(token_counts)//len(token_counts)}")
print(f"Model limit: {embedder.max_seq_length}")

over = [c for c, t in zip(all_chunks, token_counts) if t > embedder.max_seq_length]
print(f"\nChunks exceeding the limit: {len(over)}")

if over:
    print("WARNING - these chunks will be truncated, losing information:")
    for c in over[:5]:
        print(f"  {c['id']}")
else:
    print("OK - all chunks fit within the model limit.")

chars_per_token = sum(lengths) / sum(token_counts)
print(f"\nThis dataset: {chars_per_token:.2f} chars per token")
print(f"Practical max chunk_size: ~{int(embedder.max_seq_length * chars_per_token)} chars")

Token count -> min: 34, max: 185, avg: 84
Model limit: 256

Chunks exceeding the limit: 0
OK - all chunks fit within the model limit.

This dataset: 4.74 chars per token
Practical max chunk_size: ~1213 chars


### Does semantic search actually work?

This is the claim RAG rests on, so it's worth testing directly rather than taking on faith. The second sentence below shares **no content words at all** with the query — "refund" vs "money back", "damaged" vs "broken", "products" vs "item". A keyword search scores it zero.

Two things to notice in the output:

- Nothing scores above 0.9, even when meaning is nearly identical. **Similarity scores are relative, not absolute** — read the ordering, not the number. This is exactly why retrieval takes top-K rather than applying a fixed score threshold.
- The third sentence shares the word "refund" with the query yet scores *lower* than the paraphrase that shares nothing. Word overlap isn't what's being measured.

In [20]:
from sentence_transformers.util import cos_sim

test_texts = [
    "What is the refund policy for damaged products?",   # the query
    "How do I get money back for a broken item?",        # same meaning, no shared words
    "Refunds are processed within 5-7 working days.",    # shares "refund", different intent
    "The employee handbook describes leave policy.",     # unrelated
]

vecs = embedder.encode(test_texts)

print(f"QUERY: {test_texts[0]}\n")
for i in range(1, len(test_texts)):
    print(f"  {cos_sim(vecs[0], vecs[i]).item():.4f}  |  {test_texts[i]}")

QUERY: What is the refund policy for damaged products?

  0.5859  |  How do I get money back for a broken item?
  0.4236  |  Refunds are processed within 5-7 working days.
  0.1033  |  The employee handbook describes leave policy.


In [21]:
# These vectors come out already normalised (norm = 1.0), which makes
# cosine similarity and dot product mathematically identical here.
# Relevant when choosing a distance metric for the vector store.
v = embedder.encode(all_chunks[0]["text"])
print(f"Vector shape: {v.shape}")
print(f"Vector norm:  {np.linalg.norm(v):.4f}")

Vector shape: (384,)
Vector norm:  1.0000


---
# 4. Indexing in ChromaDB

Embeddings need somewhere to live that can search them quickly. A vector database indexes them using an approximate nearest neighbour structure — HNSW here — so search stays fast as the collection grows, instead of comparing the query against every single vector.

`PersistentClient` writes to disk, so the index survives a kernel restart. That matters because embedding is the slowest step in the setup, and you don't want to repeat it every session.

In [22]:
import chromadb

client = chromadb.PersistentClient(path=CHROMA_PATH)

# Start clean so re-running the notebook doesn't duplicate entries
try:
    client.delete_collection(COLLECTION_NAME)
    print("Existing collection deleted.")
except Exception:
    pass

collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"},
)

print(f"Collection '{COLLECTION_NAME}' created.")

Collection 'pawan_docs' created.


The metadata stored alongside each chunk — source, type, position — is what makes citations possible later, and what would let you filter a search to a single document if you needed to.

In [23]:
import time

texts = [c["text"] for c in all_chunks]
ids = [c["id"] for c in all_chunks]
metadatas = [
    {"source": c["source"], "doc_type": c["doc_type"], "chunk_index": c["chunk_index"]}
    for c in all_chunks
]

print(f"Embedding {len(texts)} chunks...")
t0 = time.time()
embeddings = embedder.encode(texts, show_progress_bar=True, batch_size=32)
elapsed = time.time() - t0

print(f"\nDone in {elapsed:.1f}s  ({len(texts)/elapsed:.0f} chunks/sec)")
print(f"Shape: {embeddings.shape}")

Embedding 166 chunks...


Batches:   0%|          | 0/6 [00:00<?, ?it/s]


Done in 1.0s  (172 chunks/sec)
Shape: (166, 384)


In [24]:
collection.add(
    ids=ids,
    embeddings=embeddings.tolist(),
    documents=texts,
    metadatas=metadatas,
)

print(f"Indexed: {collection.count()} chunks")

Indexed: 166 chunks


---
# 5. Retrieval

The query goes through the *same* embedding model used for indexing — a hard requirement, because two different models produce vectors in different spaces, and comparing across them yields meaningless scores.

ChromaDB returns cosine *distance*; the function below converts it to similarity (1 − distance) simply because it reads more naturally: 1 means a perfect match.

In [25]:
def retrieve(query, top_k=FINAL_K):
    """Stage 1: fast approximate search over the whole collection."""
    query_vec = embedder.encode(query).tolist()

    results = collection.query(query_embeddings=[query_vec], n_results=top_k)

    hits = []
    for i in range(len(results["ids"][0])):
        hits.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "similarity": 1 - results["distances"][0][i],
        })
    return hits

In [26]:
def show(query, hits, score_key="similarity", width=120):
    print("=" * 100)
    print(f"QUERY: {query}\n")
    for i, h in enumerate(hits, 1):
        preview = h["text"].replace("\n", " ")[:width]
        print(f"[{i}] {h[score_key]:>8.3f} | {h['metadata']['source'][:30]:30} | {preview}")
    print()


show("What is the refund policy for damaged products?",
     retrieve("What is the refund policy for damaged products?"))

QUERY: What is the refund policy for damaged products?

[1]    0.654 | Refund_and_Returns_Policy.pdf  | [Refund and Returns Policy] this document. Nothing in this policy limits the statutory rights of a consumer under the Co
[2]    0.629 | Refund_and_Returns_Policy.pdf  | [Refund and Returns Policy] 4.3 Partial Refunds A partial refund may be issued where a product is returned in a conditio
[3]    0.617 | Refund_and_Returns_Policy.pdf  | [Refund and Returns Policy] • Consumables including lubricants, seals, and filters, once the seal is broken. • Products 
[4]    0.613 | Refund_and_Returns_Policy.pdf  | [Refund and Returns Policy] 7.1 Eligibility A product may be exchanged for a different model within 30 days of delivery,
[5]    0.599 | Refund_and_Returns_Policy.pdf  | [Refund and Returns Policy] • Damage caused by incorrect installation, where installation was not performed by the compa



### A debugging lesson: truncated previews lie

On the SP-1101 query, the top result looks wrong:

> *"When contacting support, customers should have the product code and serial number available..."*

That reads like a support instruction, not a parts table. The obvious conclusion is that retrieval failed and the query needs hybrid search to match the exact code.

That conclusion would be wrong. Print the full chunk before believing a preview.

In [27]:
hits = retrieve("What is compatible with SP-1101?", top_k=3)

print("=== WHAT THE PREVIEW SHOWS ===")
print(hits[0]["text"].replace("\n", " ")[:110])

print("\n\n=== WHAT THE CHUNK ACTUALLY CONTAINS ===")
print(hits[0]["text"])

print(f"\n\nSP-1101 present in this chunk: {'SP-1101' in hits[0]['text']}")

=== WHAT THE PREVIEW SHOWS ===
[Product Catalogue 2025-26] Part code | Description | Compatible with | Price (Rs) SP-3302 | Bearing set, heav


=== WHAT THE CHUNK ACTUALLY CONTAINS ===
[Product Catalogue 2025-26]
Part code | Description | Compatible with | Price (Rs)
SP-3302 | Bearing set, heavy duty | PX-3600, SB-3000 | 2,340
SP-4401 | Control card, CP-300 | CP-300 | 7,800
SP-4402 | Display module, CP-500V | CP-500V | 9,600
SP-5501 | Pressure transducer, 0-10 bar | CP-500V | 5,200
SP-5502 | Dry-run sensor probe | CP-100, CP-300 | 1,450


SP-1101 present in this chunk: False


Depending on your chunk settings, the answer may have been sitting in the retrieved chunk the whole time, just past the 110-character preview cutoff.

Two things follow from this:

- **Judging retrieval from truncated output produces false diagnoses.** The fix here was in chunking, not in retrieval — but that only became visible after printing the full text.
- **Retrieval succeeding isn't the same as retrieval succeeding *well*.** Even when the right chunk ranks first, a similarity score around 0.44 on a chunk that literally contains the queried string is weak. That gap is the embedding being diluted across two topics — which is what the table separation in Section 2.5 addresses.

### Three queries that each stress something different

Run these and read the results carefully. The previews are truncated, so when a result looks wrong, print the full chunk before concluding anything — a chunk's most relevant content is often past the preview cutoff.

1. **Annual leave** — a straightforward semantic match. This is what good retrieval looks like: high scores, all from the right document.
2. **SP-1101** — an exact product code. Bi-encoders struggle here, because `SP-1101` and `SP-3302` look nearly identical in embedding space: same pattern, same structure, only the digits differ. Watch whether the right table chunk actually ranks first.
3. **"Can I work from home?"** — the phrase appears nowhere in the documents, which say "hybrid working" and "remote working" instead. Expect low scores across the board and unreliable ordering.

In [28]:
for q in [
    "How many days of annual leave do employees get?",
    "What is compatible with SP-1101?",
    "Can I work from home?",
]:
    show(q, retrieve(q, top_k=3))

QUERY: How many days of annual leave do employees get?

[1]    0.760 | Employee_Handbook.docx         | [Employee Handbook] 21 days in advance for leave exceeding three days. Up to 15 days of unused annual leave may be carri
[2]    0.739 | Employee_Handbook.docx         | [Employee Handbook] 4.1 Annual Leave Confirmed employees are entitled to 21 days of paid annual leave per calendar year,
[3]    0.696 | Employee_Handbook.docx         | [Employee Handbook] of the year may be offset against annual leave, or taken as unpaid leave with the approval of the de

QUERY: What is compatible with SP-1101?

[1]    0.518 | Product_Catalogue_2025-26.docx | [Product Catalogue 2025-26] Part code | Description | Compatible with | Price (Rs) SP-3302 | Bearing set, heavy duty | P
[2]    0.459 | Product_Catalogue_2025-26.docx | [Product Catalogue 2025-26] Part code | Description | Compatible with | Price (Rs) SP-1101 | Mechanical seal kit, 25 mm 
[3]    0.362 | Product_Catalogue_2025-26.docx | [Product 

---
# 6. Reranking

The retriever above is a **bi-encoder**: query and documents are encoded independently, then compared. That's what makes it fast enough to search a whole collection — documents are embedded once, ahead of time — but it also means the query and document never actually "see" each other, so fine distinctions get missed.

A **cross-encoder** feeds the query and one document through the model *together*, so it can directly weigh them against each other. Far more accurate, and far too slow to run against an entire collection — there's nothing to pre-compute.

Hence two stages: the bi-encoder casts a wide net cheaply (20 candidates), the cross-encoder re-scores just those (keeping 5). Recall is stage 1's job; precision is stage 2's.

**The consequence is worth internalising: reranking can only reorder what stage 1 already found.** If the right chunk never made the top 20, no amount of reranking will surface it.

In [29]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(RERANKER_MODEL)
print(f"Reranker loaded: {RERANKER_MODEL}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [30]:
def retrieve_with_rerank(query, retrieve_k=RETRIEVE_K, final_k=FINAL_K):
    """Stage 1 (wide, fast) followed by stage 2 (narrow, accurate)."""
    candidates = retrieve(query, top_k=retrieve_k)

    pairs = [[query, c["text"]] for c in candidates]
    scores = reranker.predict(pairs)

    for c, s in zip(candidates, scores):
        c["rerank_score"] = float(s)

    return sorted(candidates, key=lambda c: c["rerank_score"], reverse=True)[:final_k]

### Compare before and after

Cross-encoder scores are raw logits, not probabilities — **negative values are normal and meaningful**. A large positive score means confidently relevant. Scores clustered tightly around −11 mean the opposite: nothing here is relevant, and the ordering among them is essentially noise.

That signal turns out to be useful on its own, and Section 8 puts it to work.

In [31]:
def compare(query, k=FINAL_K):
    print("=" * 100)
    print(f"QUERY: {query}\n")

    print("--- STAGE 1 ONLY (bi-encoder) ---")
    for i, h in enumerate(retrieve(query, top_k=k), 1):
        preview = h["text"].replace("\n", " ")[:110]
        print(f"[{i}] {h['similarity']:>7.3f} | {h['metadata']['source'][:28]:28} | {preview}")

    print("\n--- STAGE 1 + 2 (reranked) ---")
    for i, h in enumerate(retrieve_with_rerank(query), 1):
        preview = h["text"].replace("\n", " ")[:110]
        print(f"[{i}] {h['rerank_score']:>+7.3f} | {h['metadata']['source'][:28]:28} | {preview}")
    print()


compare("What is compatible with SP-1101?")

QUERY: What is compatible with SP-1101?

--- STAGE 1 ONLY (bi-encoder) ---
[1]   0.518 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] Part code | Description | Compatible with | Price (Rs) SP-3302 | Bearing set, heav
[2]   0.459 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] Part code | Description | Compatible with | Price (Rs) SP-1101 | Mechanical seal k
[3]   0.362 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] are identified by a part code distinct from the product code. Part codes follow th
[4]   0.303 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] 2.2 PX-2400 Product code: PX-2400 A mid-range single-stage pump for general indust
[5]   0.289 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] 2.1 PX-1200 Product code: PX-1200 A compact single-stage pump for light-duty water

--- STAGE 1 + 2 (reranked) ---
[1]  +5.462 | Product_Catalogue_2025-26.do | [Product Catalogue 2025-26] Part code | Description | Compatible wit

On the SP-1101 query, the bi-encoder typically ranks the *wrong* table chunk first — the two look almost identical to it. The cross-encoder fixes it immediately and with a wide score gap, because it can see whether the string `SP-1101` is literally present in the chunk it's scoring.

Now compare the other two queries. Reranking helps clearly on one and barely at all on the other — which is the honest outcome, not a failure of the technique.

In [32]:
compare("What is the refund policy for damaged products?")

QUERY: What is the refund policy for damaged products?

--- STAGE 1 ONLY (bi-encoder) ---
[1]   0.654 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] this document. Nothing in this policy limits the statutory rights of a consumer un
[2]   0.629 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] 4.3 Partial Refunds A partial refund may be issued where a product is returned in 
[3]   0.617 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] • Consumables including lubricants, seals, and filters, once the seal is broken. •
[4]   0.613 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] 7.1 Eligibility A product may be exchanged for a different model within 30 days of
[5]   0.599 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] • Damage caused by incorrect installation, where installation was not performed by

--- STAGE 1 + 2 (reranked) ---
[1]  +6.448 | Refund_and_Returns_Policy.pd | [Refund and Returns Policy] 4.3 Partial Refunds A par

In [33]:
compare("Can I work from home?")

QUERY: Can I work from home?

--- STAGE 1 ONLY (bi-encoder) ---
[1]   0.350 | IT_Security_Policy.pdf       | [IT Security Policy] 4.2 Remote Access Remote access to internal systems is permitted only through the corpora
[2]   0.304 | Travel_and_Expense_Policy.do | [Travel and Expense Policy] 6.1 Communication Roaming charges incurred on international travel are reimbursed 
[3]   0.296 | Employee_Handbook.docx       | [Employee Handbook] 2.3 Flexible Working Employees who have completed probation may request a flexible working
[4]   0.286 | IT_Security_Policy.pdf       | [IT Security Policy] 4.3 Public Wi-Fi Where a user must work from a public Wi-Fi network, the VPN must be acti
[5]   0.279 | Employee_Handbook.docx       | [Employee Handbook] 3.4 Use of Company Property Company equipment, including laptops, mobile devices, vehicles

--- STAGE 1 + 2 (reranked) ---
[1]  -7.352 | Travel_and_Expense_Policy.do | [Travel and Expense Policy] 6.1 Communication Roaming charges incurred on inter

---
# 7. Generation

The retrieved chunks get assembled into a prompt alongside the question, and a locally-running LLM produces the answer.

Four choices in the prompt below, each doing specific work:

- **"Use ONLY the context"** — grounding. Without it the model blends retrieved facts with its own training data, and you lose the ability to tell which is which.
- **An explicit way to say "I don't know"** — without permission to refuse, a model will produce *something*, because being helpful is its default. The escape hatch has to be stated.
- **Source citations** — traceability, which is one of RAG's main advantages over fine-tuning.
- **"Quote exact numbers"** — these documents are full of figures (30 days, 48 hours, Rs 1,500) that a model will otherwise paraphrase into something subtly wrong.

`temperature=0.1` serves the same goal: low temperature keeps the model close to the supplied text instead of embellishing it.

In [34]:
import requests

def ask_llm(prompt, temperature=TEMPERATURE):
    """Send a prompt to the local Ollama server."""
    r = requests.post(
        OLLAMA_URL,
        json={
            "model": LLM_MODEL,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": temperature},
        },
        timeout=300,
    )
    r.raise_for_status()
    return r.json()["response"].strip()


print(ask_llm("Reply with exactly: OK"))

OK


In [35]:
def build_prompt(query, chunks):
    """Assemble retrieved chunks and the question into one prompt."""
    context = "\n\n---\n\n".join(
        f"[Source {i}: {c['metadata']['source']}]\n{c['text']}"
        for i, c in enumerate(chunks, 1)
    )

    return f"""You are a helpful assistant answering questions about Pawan Pvt Ltd's internal documents.

Answer the question using ONLY the context provided below. Follow these rules:
- If the context does not contain the answer, say "I don't have that information in the provided documents."
- Do not use any knowledge outside the context.
- Cite the source number for each fact, like [Source 2].
- Be concise and specific. Quote exact numbers, dates, and terms from the context.

CONTEXT:
{context}

QUESTION: {query}

ANSWER:"""

In [36]:
def rag_answer(query, rerank=True, verbose=True):
    """Full pipeline: retrieve -> rerank -> prompt -> generate."""
    t0 = time.time()
    chunks = retrieve_with_rerank(query) if rerank else retrieve(query, top_k=FINAL_K)
    t_retrieval = time.time() - t0

    prompt = build_prompt(query, chunks)

    t1 = time.time()
    answer = ask_llm(prompt)
    t_generation = time.time() - t1

    if verbose:
        print(f"QUESTION: {query}\n")
        print(f"ANSWER:\n{answer}\n")
        print("-" * 80)
        print("SOURCES:")
        for i, c in enumerate(chunks, 1):
            score = c.get("rerank_score", c.get("similarity"))
            print(f"  [{i}] {c['metadata']['source']}  ({score:+.3f})")
        print(f"\nTIMING: retrieval {t_retrieval:.2f}s | generation {t_generation:.2f}s "
              f"| total {t_retrieval + t_generation:.2f}s")

    return {"answer": answer, "chunks": chunks, "prompt": prompt,
            "t_retrieval": t_retrieval, "t_generation": t_generation}

In [37]:
result = rag_answer("What is the refund policy for damaged products?")

QUESTION: What is the refund policy for damaged products?

ANSWER:
According to the Refund and Returns Policy, the refund policy for damaged products is as follows:

* For products that arrive damaged, a report must be made within 48 hours of delivery [Source 3].
* If the damage is discovered only after installation, and the customer can demonstrate that the damage predates installation, the claim is assessed under the warranty procedure in Section 5 [Source 4].
* Damage caused by incorrect installation, operation outside published specification, unauthorised repair or modification, or cosmetic damage that does not affect function is not covered [Source 5].

Additionally, a partial refund may be issued for products returned in a condition that is not resaleable but is not defective, with deductions for missing or damaged original packaging or missing accessories or documentation [Source 1].

--------------------------------------------------------------------------------
SOURCES:
  [1]

### Read the timing numbers

Generation typically accounts for around 85% of total latency here, with retrieval — reranking included — making up the rest. Embedding the query itself is a rounding error.

That ordering has a direct practical consequence: optimising retrieval speed buys you very little. Streaming the response, shortening the output, or using a smaller model are where the real gains are.

In [38]:
result = rag_answer("How many days of annual leave do employees get?")

QUESTION: How many days of annual leave do employees get?

ANSWER:
According to [Source 1], employees are entitled to 21 days of paid annual leave per calendar year.

--------------------------------------------------------------------------------
SOURCES:
  [1] Employee_Handbook.docx  (+9.191)
  [2] Employee_Handbook.docx  (+7.817)
  [3] Employee_Handbook.docx  (+6.553)
  [4] Employee_Handbook.docx  (+4.756)
  [5] Employee_Handbook.docx  (+2.741)

TIMING: retrieval 0.89s | generation 6.48s | total 7.37s


In [39]:
result = rag_answer("What is compatible with SP-1101?")

QUESTION: What is compatible with SP-1101?

ANSWER:
According to [Source 1], SP-1101 is a Mechanical seal kit, 25 mm, and it is compatible with PX-1200. [Source 1: Product_Catalogue_2025-26.docx]

--------------------------------------------------------------------------------
SOURCES:
  [1] Product_Catalogue_2025-26.docx  (+5.462)
  [2] Product_Catalogue_2025-26.docx  (+0.919)
  [3] Product_Catalogue_2025-26.docx  (-7.201)
  [4] Product_Catalogue_2025-26.docx  (-10.314)
  [5] Product_Catalogue_2025-26.docx  (-10.344)

TIMING: retrieval 0.40s | generation 10.51s | total 10.91s


### The grounding test

The model knows the capital of France perfectly well from its training data. If the prompt is doing its job, it will refuse to answer anyway — because that fact isn't in the retrieved context.

This single test is the clearest demonstration of what RAG actually changes about a model's behaviour.

In [40]:
result = rag_answer("What is the capital of France?")

QUESTION: What is the capital of France?

ANSWER:
I don't have that information in the provided documents.

--------------------------------------------------------------------------------
SOURCES:
  [1] Travel_and_Expense_Policy.docx  (-11.053)
  [2] Product_Catalogue_2025-26.docx  (-11.057)
  [3] Product_Catalogue_2025-26.docx  (-11.064)
  [4] Product_Catalogue_2025-26.docx  (-11.075)
  [5] Product_Catalogue_2025-26.docx  (-11.076)

TIMING: retrieval 1.11s | generation 6.56s | total 7.67s


---
# 8. Using the Reranker as a Relevance Gate

Look at the rerank scores on that last query: all clustered around −11, barely distinguishable from each other. The cross-encoder is saying plainly that nothing in the collection is relevant.

That's actionable. If the top rerank score is below zero, there's no point calling the LLM at all — skip straight to "I don't have that information."

Two things this buys you:

- **Latency and cost.** An entire generation call, several seconds of it, avoided.
- **Safety.** A model that never sees the prompt cannot hallucinate an answer to it.

This is query routing in its simplest useful form — deciding whether the expensive step is worth taking, instead of always taking it.

In [41]:
RELEVANCE_THRESHOLD = 0.0   # top rerank score must clear this

def rag_answer_gated(query, threshold=RELEVANCE_THRESHOLD, verbose=True):
    """Skip the LLM entirely when nothing retrieved looks relevant."""
    chunks = retrieve_with_rerank(query)
    top_score = chunks[0]["rerank_score"]

    if top_score < threshold:
        if verbose:
            print(f"QUESTION: {query}\n")
            print("ANSWER:\nI don't have that information in the provided documents.\n")
            print(f"(LLM skipped - top rerank score {top_score:+.2f} below threshold {threshold})")
        return {"answer": None, "skipped": True, "top_score": top_score}

    return rag_answer(query, verbose=verbose)

In [42]:
rag_answer_gated("What is the capital of France?")

QUESTION: What is the capital of France?

ANSWER:
I don't have that information in the provided documents.

(LLM skipped - top rerank score -11.05 below threshold 0.0)


{'answer': None, 'skipped': True, 'top_score': -11.052515983581543}

In [43]:
rag_answer_gated("What is the refund policy for damaged products?")

QUESTION: What is the refund policy for damaged products?

ANSWER:
According to the Refund and Returns Policy, the refund policy for damaged products is as follows:

* For products that arrive damaged, a report must be made within 48 hours of delivery (Source 3).
* If the damage is discovered only after installation, and the customer can demonstrate that the damage predates installation, the claim is assessed under the warranty procedure in Section 5 (Source 4).
* Damage caused by incorrect installation, operation outside the published specification, unauthorised repair or modification, or cosmetic damage that does not affect function is not covered (Source 5).

Additionally, a partial refund may be issued for products that are returned in a condition that is not resaleable but is not defective, with deductions for missing or damaged original packaging or missing accessories or documentation (Source 1).

--------------------------------------------------------------------------------
S

{'answer': 'According to the Refund and Returns Policy, the refund policy for damaged products is as follows:\n\n* For products that arrive damaged, a report must be made within 48 hours of delivery (Source 3).\n* If the damage is discovered only after installation, and the customer can demonstrate that the damage predates installation, the claim is assessed under the warranty procedure in Section 5 (Source 4).\n* Damage caused by incorrect installation, operation outside the published specification, unauthorised repair or modification, or cosmetic damage that does not affect function is not covered (Source 5).\n\nAdditionally, a partial refund may be issued for products that are returned in a condition that is not resaleable but is not defective, with deductions for missing or damaged original packaging or missing accessories or documentation (Source 1).',
 'chunks': [{'id': 'Refund_and_Returns_Policy.pdf::chunk_10',
   'text': '[Refund and Returns Policy]\n4.3 Partial Refunds A parti

### Choosing the threshold is a real trade-off

Zero is a starting point, not a correct answer. Set it too high and legitimate questions get refused; too low and the gate stops catching anything.

The "Can I work from home?" query sits right in the grey zone: the relevant section genuinely exists in the documents, but retrieval doesn't find it confidently, so scores land somewhere around −7 to −9. A threshold of 0 refuses it. Whether that's the right call depends on whether a wrong answer or a refusal is worse for your use case.

Tuning this by feel is guesswork. Measuring it is what Part 9 covers.

In [44]:
for q in [
    "Can I work from home?",
    "What is the warranty period for control panels?",
    "How do I claim travel expenses?",
    "What is the CEO's salary?",          # not in any document
]:
    r = retrieve_with_rerank(q)
    top = r[0]["rerank_score"]
    verdict = "ANSWER" if top >= RELEVANCE_THRESHOLD else "REFUSE"
    print(f"{verdict:7} | {top:+8.3f} | {q}")

REFUSE  |   -7.352 | Can I work from home?
ANSWER  |   +7.742 | What is the warranty period for control panels?
ANSWER  |   +5.010 | How do I claim travel expenses?
REFUSE  |   -7.908 | What is the CEO's salary?


---
# 9. Experiments Worth Running

The pipeline works. What follows is how you'd actually improve it — by changing one variable and observing the effect, rather than changing several and guessing which one mattered.

Each of these maps to a concept from the concepts phase of the series.

### Does reranking actually earn its latency?

It costs a second or two per query. Run the same question with and without it and judge whether the retrieved set genuinely improved — on some queries it transforms the results, on others it changes nothing.

In [45]:
q = "What is compatible with SP-1101?"

print(">>> WITHOUT RERANKING")
r1 = rag_answer(q, rerank=False)

print("\n" + "=" * 80 + "\n")

print(">>> WITH RERANKING")
r2 = rag_answer(q, rerank=True)

>>> WITHOUT RERANKING
QUESTION: What is compatible with SP-1101?

ANSWER:
According to [Source 2], SP-1101 is a Mechanical seal kit, 25 mm, and it is compatible with PX-1200. [Source 2: SP-1101 | Mechanical seal kit, 25 mm | PX-1200 | 1,240]

--------------------------------------------------------------------------------
SOURCES:
  [1] Product_Catalogue_2025-26.docx  (+0.518)
  [2] Product_Catalogue_2025-26.docx  (+0.459)
  [3] Product_Catalogue_2025-26.docx  (+0.362)
  [4] Product_Catalogue_2025-26.docx  (+0.303)
  [5] Product_Catalogue_2025-26.docx  (+0.289)

TIMING: retrieval 0.44s | generation 10.25s | total 10.69s


>>> WITH RERANKING
QUESTION: What is compatible with SP-1101?

ANSWER:
According to [Source 1], SP-1101 is a Mechanical seal kit, 25 mm, and it is compatible with PX-1200. [Source 1: Product_Catalogue_2025-26.docx]

--------------------------------------------------------------------------------
SOURCES:
  [1] Product_Catalogue_2025-26.docx  (+5.462)
  [2] Product_Cat

### How much does chunk size change retrieval?

Rebuild the index with a different `CHUNK_SIZE` and compare. Smaller chunks are more precise but risk splitting an answer across two of them; larger chunks hold more context but blur the embedding across multiple topics.

Remember the ceiling measured in Section 3 — going past roughly 700 characters here means silent truncation at the embedding model.

In [46]:
def rebuild_index(chunk_size, overlap=OVERLAP):
    """Rebuild chunks and index with a different chunk size."""
    global all_chunks, collection

    all_chunks = []
    for doc in documents:
        for i, ch in enumerate(chunk_document(doc, chunk_size, overlap)):
            all_chunks.append({
                "id": f"{doc['source']}::chunk_{i}",
                "text": ch,
                "source": doc["source"],
                "doc_type": doc["type"],
                "chunk_index": i,
            })

    try:
        client.delete_collection(COLLECTION_NAME)
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"}
    )

    embs = embedder.encode([c["text"] for c in all_chunks], batch_size=32)
    collection.add(
        ids=[c["id"] for c in all_chunks],
        embeddings=embs.tolist(),
        documents=[c["text"] for c in all_chunks],
        metadatas=[{"source": c["source"], "doc_type": c["doc_type"],
                    "chunk_index": c["chunk_index"]} for c in all_chunks],
    )

    lens = [len(c["text"]) for c in all_chunks]
    print(f"chunk_size={chunk_size}: {len(all_chunks)} chunks, "
          f"avg {sum(lens)//len(lens)} chars")


# Try a few sizes and watch how retrieval shifts
# rebuild_index(300)
# show("What is the refund policy for damaged products?",
#      retrieve_with_rerank("What is the refund policy for damaged products?"),
#      score_key="rerank_score")

# Restore the default when finished
# rebuild_index(CHUNK_SIZE)

### Does the number of retrieved chunks matter?

More chunks mean more context — and more noise, more tokens, and higher cost. Somewhere there's a point where adding chunks stops helping and starts hurting.

In [47]:
q = "What is the refund policy for damaged products?"

for k in [1, 3, 5, 10]:
    chunks = retrieve_with_rerank(q, final_k=k)
    prompt = build_prompt(q, chunks)
    t0 = time.time()
    answer = ask_llm(prompt)
    dt = time.time() - t0

    print(f"=== final_k={k}  ({len(prompt)} prompt chars, {dt:.1f}s) ===")
    print(answer)
    print()

=== final_k=1  (999 prompt chars, 8.7s) ===
According to the Refund and Returns Policy [Source 1], a partial refund may be issued for a product that is returned in a condition that is not resaleable but is not defective. In such cases, deductions are made as follows:

* Missing or damaged original packaging: 10% of invoice value [Source 1: 4.3].
* Minor handling marks do not affect the refund amount, but the product's condition is not resaleable.

Note that the policy does not specify a refund amount for damaged products, but rather for products that are not resaleable due to minor handling marks.

=== final_k=3  (1928 prompt chars, 7.3s) ===
According to [Source 3], if a product arrives damaged, a report must be made within 48 hours of delivery. If the report is made after 48 hours, it may be refused. The customer must submit photographs of the outer packaging, the damaged product, and the shipping label through the support portal or by email to claims@pawanpvtltd.example. [Source 3]


### Does temperature change grounding?

Low temperature keeps the model close to the supplied text. Raise it and watch whether answers start drifting from what the context actually says.

In [48]:
q = "What is the refund policy for damaged products?"
chunks = retrieve_with_rerank(q)
prompt = build_prompt(q, chunks)

for temp in [0.0, 0.5, 1.0]:
    print(f"=== temperature={temp} ===")
    print(ask_llm(prompt, temperature=temp))
    print()

=== temperature=0.0 ===
According to the Refund and Returns Policy, the refund policy for damaged products is as follows:

* For products that arrive damaged, a report must be made within 48 hours of delivery [Source 3].
* If the damage is discovered after installation, and the customer can demonstrate that the damage predates installation, the claim is assessed under the warranty procedure in Section 5 [Source 4].
* Damage caused by incorrect installation, operation outside published specification, unauthorised repair or modification, or cosmetic damage that does not affect function is not covered [Source 5].

Additionally, a partial refund may be issued for products that are returned in a condition that is not resaleable but is not defective, with deductions for missing or damaged original packaging or missing accessories or documentation [Source 1].

=== temperature=0.5 ===
According to the Refund and Returns Policy documents, the refund policy for damaged products is as follows:

*

---
## Where This Goes Next

This pipeline handles straightforward questions well. Its limits are worth naming honestly:

- **Exact identifiers depend on luck.** Reranking rescued the `SP-1101` query, but only because the right chunk happened to reach the top 20. On a corpus of ten thousand chunks rather than a hundred and sixty, it wouldn't have — and reranking can't retrieve what stage 1 never found. That's the case for hybrid search.
- **Nothing here is measured.** Every judgement in this notebook — is this chunk size better, did reranking help, is the threshold right — has been made by reading output and forming an impression. That doesn't scale, and it isn't reliable.

Both are the subject of **Part 9: Hybrid Search and Evaluation**.

📚 [Full series on AWS Builder Center](https://builder.aws.com/community/@pabbico)